# Feature Inspection

This notebook lets you:
1. Run feature extraction and see the output
2. Load an existing `features.csv` and concat new patients onto it
3. Check NaN counts per subject and per feature
4. See exactly which patients are incomplete and what they're missing

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import gzip, shutil
import os

## Settings
Edit these paths to match your setup.

In [3]:
MERGED_DIR     = Path("C:/Users/AKLO0022/EOG_REM/merged_csv_eog")      
FEATURE_CSV    = Path("C:/Users/AKLO0022/EOG_REM/features_csv/features.csv") 
PATIENT_EXCEL  = Path("L:/Auditdata/RBD PD/PD-RBD Glostrup Database_ok/GlostrupRBDData.xlsx")  
LIGHTS_DIR     = Path("L:/Auditdata/RBD PD/PD-RBD Glostrup Database_ok") 
FS             = 250.0
PATTERN        = "*contiguous_eog_merged.csv"

---
## 1. Compress merged CSVs

Run this to compress all merged CSVs in `merged_csv_eog` and delete the uncompressed versions. This is optional but saves a lot of disk space. You can still read the compressed CSVs with `pd.read_csv(..., compression='gzip')`

In [27]:
# compress_merged.py  (delete after running)

for f in MERGED_DIR.glob(PATTERN):
    gz = f.with_suffix(".csv.gz")
    with open(f, "rb") as fi, gzip.open(gz, "wb", 6) as fo:
        shutil.copyfileobj(fi, fo)
    saved = f.stat().st_size - gz.stat().st_size
    print(f"{f.name}: freed {saved / 1024**2:.1f} MB")
    f.unlink()

DCSM_127_a_contiguous_eog_merged.csv: freed 1647.1 MB
DCSM_129_a_contiguous_eog_merged.csv: freed 1063.0 MB
DCSM_12_a_contiguous_eog_merged.csv: freed 1090.5 MB
DCSM_136_a_contiguous_eog_merged.csv: freed 1597.3 MB
DCSM_13_a_contiguous_eog_merged.csv: freed 1463.2 MB
DCSM_141_a_contiguous_eog_merged.csv: freed 1427.0 MB
DCSM_143_a_contiguous_eog_merged.csv: freed 1470.3 MB
DCSM_147_a_contiguous_eog_merged.csv: freed 1734.6 MB
DCSM_149_a_contiguous_eog_merged.csv: freed 1414.9 MB
DCSM_14_a_contiguous_eog_merged.csv: freed 1257.2 MB
DCSM_153_a_contiguous_eog_merged.csv: freed 1369.9 MB
DCSM_168_a_contiguous_eog_merged.csv: freed 1212.0 MB
DCSM_169_a_contiguous_eog_merged.csv: freed 1481.1 MB
DCSM_16_a_contiguous_eog_merged.csv: freed 1038.9 MB


---
## 2. Check lights txt
Check for lights.txt files with 0 duration.

In [12]:
problems = []
for lights_file in sorted(LIGHTS_DIR.rglob("lights.txt")):
    try:
        txt = pd.read_csv(lights_file)
        txt.columns = [c.strip().lower() for c in txt.columns]

        off = txt["lights_off"].iloc[0]
        on  = txt["lights_on"].iloc[0]

        # Check for NaN or zero values 
        off_zero = pd.isna(off) or float(off) == 0.0
        on_zero = pd.isna(on) or float(on) == 0.0

        if off_zero and on_zero:
            problems.append({"session": lights_file.parent.name, "lights_off": off, "lights_on": on, "issue": "both zero"})
        elif off_zero:
            problems.append({"session": lights_file.parent.name, "lights_off": off, "lights_on": on, "issue": "lights_off is zero"})
        elif on_zero:
            problems.append({"session": lights_file.parent.name, "lights_off": off, "lights_on": on, "issue": "lights_on is zero"})
    except Exception as e:
        problems.append({"session": lights_file.parent.name, "lights_off": None, "lights_on": None, "issue": str(e)})

df_lights = pd.DataFrame(problems)
print(f"Problematic lights files: {len(df_lights)} / {len(list(LIGHTS_DIR.rglob('lights.txt')))}\n")
print(df_lights.to_string(index=False))

Problematic lights files: 3 / 223

   session  lights_off  lights_on     issue
DCSM_204_a           0          0 both zero
DCSM_353_b           0          0 both zero
DCSM_467_a           0          0 both zero


---
## 3. NaN inspection
Load `features.csv` and check for missing values.

In [4]:
df = pd.read_csv(FEATURE_CSV)
print(f"Loaded: {df.shape[0]} subjects, {df.shape[1]} columns")

Loaded: 222 subjects, 117 columns


### 3a. NaN count per feature
Which features have the most missing values?

In [5]:
nan_per_feature = df.isna().sum()
nan_per_feature = nan_per_feature[nan_per_feature > 0].sort_values(ascending=False)

if nan_per_feature.empty:
    print("No NaN values in any feature!")
else:
    print(f"{len(nan_per_feature)} features have NaN values:\n")
    for feat, count in nan_per_feature.items():
        pct = count / len(df) * 100
        print(f"  {feat:<45s}  {count:3d} / {len(df)}  ({pct:.1f}%)")

77 features have NaN values:

  rem_event_median_duration_s                     17 / 222  (7.7%)
  rem_event_mean_loc_rise_slope                   17 / 222  (7.7%)
  rem_event_mean_roc_rise_slope                   17 / 222  (7.7%)
  rem_event_mean_loc_amp_uv                       17 / 222  (7.7%)
  rem_event_mean_roc_amp_uv                       17 / 222  (7.7%)
  rem_event_mean_duration_s                       17 / 222  (7.7%)
  rem_em_mean_amp_uv                              11 / 222  (5.0%)
  rem_em_mean_duration_s                          11 / 222  (5.0%)
  eeg__n1__total                                  10 / 222  (4.5%)
  eeg__n1__theta                                  10 / 222  (4.5%)
  eeg__n1__alpha                                  10 / 222  (4.5%)
  eeg__n1__beta                                   10 / 222  (4.5%)
  eeg__n1__delta                                  10 / 222  (4.5%)
  eeg__n1__theta_ratio                            10 / 222  (4.5%)
  rem_epoch_std_duration_min    

### 3b. NaN count per subject
Which subjects have missing values, and how many?

In [6]:
id_cols = ["subject_id", "DCSM_ID"]
feature_cols = [c for c in df.columns if c not in id_cols and pd.api.types.is_numeric_dtype(df[c])]

nan_per_subject = df[feature_cols].isna().sum(axis=1)
subjects_with_nan = df[nan_per_subject > 0][["subject_id"]].copy()
subjects_with_nan["nan_count"] = nan_per_subject[nan_per_subject > 0].values

if subjects_with_nan.empty:
    print("All subjects have complete features!")
else:
    print(f"{len(subjects_with_nan)} subject(s) have NaN values:\n")
    for _, row in subjects_with_nan.sort_values("nan_count", ascending=False).iterrows():
        print(f"  {row['subject_id']:<50s}  {row['nan_count']:.0f} missing features")

36 subject(s) have NaN values:

  DCSM_10_a                                           50 missing features
  DCSM_109_a                                          50 missing features
  DCSM_94_a                                           32 missing features
  DCSM_204_a                                          20 missing features
  DCSM_292_a                                          20 missing features
  DCSM_53_a                                           16 missing features
  DCSM_129_a                                          12 missing features
  DCSM_106_a                                          12 missing features
  DCSM_410_a                                          8 missing features
  DCSM_464_a                                          8 missing features
  DCSM_7_a                                            7 missing features
  DCSM_100_a                                          6 missing features
  DCSM_353_b                                          6 missing features
  DCSM_351_

### 3c. Detailed NaN report
For each subject with NaN values, show exactly which features are missing.

In [7]:
rows_with_nan = df[df[feature_cols].isna().any(axis=1)]

if rows_with_nan.empty:
    print("No NaN values found.")
else:
    for _, row in rows_with_nan.iterrows():
        sid = row["subject_id"]
        missing = [c for c in feature_cols if pd.isna(row[c])]
        print(f"\n{sid}  ({len(missing)} missing):")
        for col in missing:
            print(f"    - {col}")


DCSM_100_a  (6 missing):
    - rem_event_mean_duration_s
    - rem_event_median_duration_s
    - rem_event_mean_loc_amp_uv
    - rem_event_mean_roc_amp_uv
    - rem_event_mean_loc_rise_slope
    - rem_event_mean_roc_rise_slope

DCSM_106_a  (12 missing):
    - eeg__n1__delta
    - eeg__n1__theta
    - eeg__n1__alpha
    - eeg__n1__beta
    - eeg__n1__total
    - eeg__n1__theta_ratio
    - eeg__n3__delta
    - eeg__n3__theta
    - eeg__n3__alpha
    - eeg__n3__beta
    - eeg__n3__total
    - eeg__n3__theta_ratio

DCSM_109_a  (50 missing):
    - total_recording_min
    - w_duration_min
    - w_fraction
    - n1_duration_min
    - n1_fraction
    - n2_duration_min
    - n2_fraction
    - n3_duration_min
    - n3_fraction
    - rem_duration_min
    - rem_fraction
    - n_rem_epochs
    - rem_epoch_count
    - rem_epoch_mean_duration_min
    - rem_epoch_std_duration_min
    - rem_epoch_min_duration_min
    - rem_epoch_max_duration_min
    - rem_event_count
    - rem_event_rate_per_min
    -

### 3d. Group label check
Check which subjects have group labels (Control, iRBD, etc.) and which don't.

In [9]:
group_cols = [c for c in ["Control", "PD(-RBD)", "PD(+RBD)", "iRBD", "PLM"] if c in df.columns]

if not group_cols:
    print("No group columns found. Run extract with --patient-excel to add them.")
else:
    has_label = df[group_cols].sum(axis=1) > 0
    n_labelled = has_label.sum()
    n_unlabelled = (~has_label).sum()
    
    print(f"With group label:     {n_labelled}")
    print(f"Without group label:  {n_unlabelled}")
    
    if n_unlabelled > 0:
        unlabelled = df.loc[~has_label, "subject_id"].tolist()
        print(f"\nUnlabelled subjects:")
        for sid in unlabelled:
            print(f"  - {sid}")
    
    # Group distribution
    print(f"\nGroup distribution:")
    for col in group_cols:
        n = (df[col] == 1).sum()
        print(f"  {col:<12s}  n = {n}")

With group label:     222
Without group label:  0

Group distribution:
  Control       n = 52
  PD(-RBD)      n = 38
  PD(+RBD)      n = 53
  iRBD          n = 38
  PLM           n = 41


---
## 4. Quick summary

In [10]:
print(f"Total subjects:       {len(df)}")
print(f"Total features:       {len(feature_cols)}")
print(f"Subjects with NaN:    {len(rows_with_nan)}")
print(f"Features with NaN:    {len(nan_per_feature)}")
print(f"Total NaN values:     {df[feature_cols].isna().sum().sum()}")

Total subjects:       222
Total features:       115
Subjects with NaN:    36
Features with NaN:    77
Total NaN values:     350


---
## 5. ID cleanup

#### Function:

In [ ]:
def clean_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    After collect_features + merge_patient_info, the DataFrame may contain:
      - subject_id  
      - DCSM_ID    
      - Duplicate group columns from patient module AND Excel join
        (Control, PD(-RBD), PD(+RBD), iRBD, PLM — possibly with _x/_y suffixes)

    This function consolidates duplicates and drops redundant columns.
    """
    df = df.copy()

    # 1) Drop DCSM_ID if it matches subject_id
    if "DCSM_ID" in df.columns:
        if df["DCSM_ID"].equals(df["subject_id"]):
            df.drop(columns=["DCSM_ID"], inplace=True)
            print("Dropped 'DCSM_ID' (identical to subject_id)")
        else:
            print("Kept 'DCSM_ID' (differs from subject_id for some rows)")

    # 2) Consolidate _x / _y suffix duplicates from pd.merge
    suffixed = {}
    for col in df.columns:
        for suffix in ("_x", "_y"):
            if col.endswith(suffix):
                base = col[: -len(suffix)]
                suffixed.setdefault(base, []).append(col)

    for base, cols in suffixed.items():
        # Keep _x, fill NaNs from _y, drop both suffixed versions
        primary = f"{base}_x"
        secondary = f"{base}_y"
        if primary in df.columns and secondary in df.columns:
            df[base] = df[primary].fillna(df[secondary])
            df.drop(columns=[primary, secondary], inplace=True)
            print(f"Merged '{primary}' + '{secondary}' → '{base}'")

    # 3) Drop any remaining _dup columns from the outer join
    dup_cols = [c for c in df.columns if c.endswith("_dup")]
    if dup_cols:
        df.drop(columns=dup_cols, inplace=True)
        print(f"Dropped _dup columns: {dup_cols}")

    print(f"\nFinal shape: {df.shape[0]} subjects × {df.shape[1]} columns")
    return df

#### Run function:

In [ ]:
df = pd.read_csv(FEATURE_CSV)                             # Load the original features CSV 
print(f"Before: {df.shape}")                              # Print shape before cleaning
df = clean_feature_columns(df)                            # Clean up redundant ID / label columns after merge
print(f"After: {df.shape}")                               # Print shape after cleaning
df.to_csv("features_csv/features_clean.csv", index=False) # Save the cleaned DataFrame to a new CSV file
print("Cleaned features saved to 'features_csv/features_clean.csv'")